# Session 1 — Fine-tune RAFT-Sintel on Spring

This notebook only trains. It starts from the supplied RAFT-Sintel checkpoint, fine-tunes on the left-camera forward-flow subset, validates on held-out sequence `0022`, and writes one portable checkpoint:

`/kaggle/working/raft-spring-left-finetuned.ckpt`

After the run completes, save a Kaggle Notebook Version with outputs and create a Kaggle Dataset containing that checkpoint. Attach that output dataset to the separate inference notebook.


In [1]:
from pathlib import Path
import importlib, json, os, platform, shutil, subprocess, sys

RUN_TRAINING = True
SEED = 3407
TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 1
ACCUMULATE_GRAD_BATCHES = 4
TRAIN_ITERS = 12
TRAIN_LR = 1.0e-5
TRAIN_WEIGHT_DECAY = 1.0e-5
TRAIN_CROP_SIZE = [540, 960]
TRAIN_PRECISION = "16-mixed"
DEVKIT_REF = "90ae81a9324c6806dc3c2482aab84a2744215bd9"

KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
DEVKIT_DIR = WORK_ROOT / "roco-spring-devkit"
TRAIN_SPRING_ROOT = KAGGLE_INPUT / "datasets/sakhawatdhrubo2/spring-train-left-10seq/spring_subset/spring"
TRAIN_MANIFEST = KAGGLE_INPUT / "datasets/sakhawatdhrubo2/spring-train-left-10seq/spring_subset/manifest.json"
PRETRAINED_CHECKPOINT = KAGGLE_INPUT / "datasets/strikingratio/ckpoint/raft-sintel-fb44381e.ckpt"
EXPECTED_TRAIN_SEQUENCES = {"0011", "0022", "0025", "0026", "0027", "0030", "0032", "0036", "0041", "0045"}
TRAIN_LOG_DIR = WORK_ROOT / "raft_spring_train_logs"
FINETUNED_CHECKPOINT = WORK_ROOT / "raft-spring-left-finetuned.ckpt"

assert KAGGLE_INPUT.is_dir(), "Run this notebook in Kaggle."
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "train_root": str(TRAIN_SPRING_ROOT),
    "epochs": TRAIN_EPOCHS,
    "iterations": TRAIN_ITERS,
    "output_checkpoint": str(FINETUNED_CHECKPOINT),
}, indent=2))


{
  "train_root": "/kaggle/input/datasets/sakhawatdhrubo2/spring-train-left-10seq/spring_subset/spring",
  "epochs": 1,
  "iterations": 12,
  "output_checkpoint": "/kaggle/working/raft-spring-left-finetuned.ckpt"
}


## 1. Preflight and verify the training pairs

Spring stores one forward flow for each adjacent frame pair, so every sequence must have exactly one more PNG than `.flo5` file. This check also confirms that sequence `0022`, the official devkit holdout, is present.


In [2]:
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True, capture_output=True, check=True,
)
print(gpu.stdout)
gpu_count = len([line for line in gpu.stdout.splitlines() if line.strip()])
if gpu_count < 1:
    raise RuntimeError("Enable a Kaggle GPU accelerator and restart the session.")

required = [TRAIN_SPRING_ROOT, PRETRAINED_CHECKPOINT] if RUN_TRAINING else []
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
if RUN_TRAINING and PRETRAINED_CHECKPOINT.stat().st_size < 1_000_000:
    raise RuntimeError("The RAFT-Sintel checkpoint appears incomplete.")

sequence_stats = {}
if RUN_TRAINING:
    train_dir = TRAIN_SPRING_ROOT / "train"
    for sequence in sorted(p for p in train_dir.iterdir() if p.is_dir()):
        frames = sorted((sequence / "frame_left").glob("frame_left_*.png"))
        flows = sorted((sequence / "flow_FW_left").glob("flow_FW_left_*.flo5"))
        if not frames or len(frames) != len(flows) + 1:
            raise RuntimeError(f"Invalid adjacent-pair counts in {sequence}: frames={len(frames)}, flows={len(flows)}")
        sequence_stats[sequence.name] = {"frames": len(frames), "flows": len(flows)}
    if set(sequence_stats) != EXPECTED_TRAIN_SEQUENCES:
        raise RuntimeError(f"Unexpected training sequences: {sorted(sequence_stats)}")
    print(json.dumps(sequence_stats, indent=2))
    print("Training pairs:", sum(v["flows"] for k, v in sequence_stats.items() if k != "0022"))
    print("Validation pairs (0022):", sequence_stats["0022"]["flows"])
    if TRAIN_MANIFEST.is_file():
        manifest = json.loads(TRAIN_MANIFEST.read_text())
        print("Manifest loaded; top-level keys:", sorted(manifest) if isinstance(manifest, dict) else type(manifest).__name__)
else:
    print("Training-data checks skipped for this inference-only run.")


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

{
  "0011": {
    "frames": 95,
    "flows": 94
  },
  "0022": {
    "frames": 19,
    "flows": 18
  },
  "0025": {
    "frames": 127,
    "flows": 126
  },
  "0026": {
    "frames": 29,
    "flows": 28
  },
  "0027": {
    "frames": 13,
    "flows": 12
  },
  "0030": {
    "frames": 126,
    "flows": 125
  },
  "0032": {
    "frames": 73,
    "flows": 72
  },
  "0036": {
    "frames": 108,
    "flows": 107
  },
  "0041": {
    "frames": 96,
    "flows": 95
  },
  "0045": {
    "frames": 198,
    "flows": 197
  }
}
Training pairs: 856
Validation pairs (0022): 18
Manifest loaded; top-level keys: ['sequences', 'sources', 'spring_root_dir_hint', 'subdirs', 'total_bytes', 'val_sequence']


## 2. Install the pinned official devkit

The same pinned revision as the original baseline is used. With Kaggle Internet disabled, attach a copy of the devkit containing `roco_spring_devkit/optical_flow/train.py`; otherwise the cell clones it.


In [3]:
def shallow_walk(root: Path, max_depth=5):
    root = root.resolve()
    for current, dirs, files in os.walk(root):
        current = Path(current)
        yield current, dirs, files
        if len(current.relative_to(root).parts) >= max_depth:
            dirs[:] = []

def find_attached_devkit():
    for current, _, files in shallow_walk(KAGGLE_INPUT):
        if "pyproject.toml" in files and (current / "roco_spring_devkit/optical_flow/train.py").is_file():
            return current
    return None

if not (DEVKIT_DIR / "roco_spring_devkit/optical_flow/train.py").is_file():
    attached = find_attached_devkit()
    if attached:
        print("Copying attached devkit:", attached)
        shutil.copytree(attached, DEVKIT_DIR, dirs_exist_ok=True)
    else:
        print("Cloning the official devkit; Kaggle Internet must be enabled.")
        subprocess.run(["git", "clone", "https://github.com/hmorimitsu/roco-spring-devkit.git", str(DEVKIT_DIR)], check=True)
if (DEVKIT_DIR / ".git").is_dir():
    subprocess.run(["git", "checkout", DEVKIT_REF], cwd=DEVKIT_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(DEVKIT_DIR)], check=True)
devkit_path = str(DEVKIT_DIR.resolve())
if devkit_path not in sys.path:
    sys.path.insert(0, devkit_path)
importlib.invalidate_caches()

from roco_spring_devkit.optical_flow.models.raft.raft import RAFT
probe = RAFT(iters=TRAIN_ITERS, corr_mode="allpairs", predict_all_directions=False)
parameter_count = sum(p.numel() for p in probe.parameters())
print(f"Devkit ready; full RAFT has {parameter_count / 1e6:.2f}M parameters")
del probe


Cloning the official devkit; Kaggle Internet must be enabled.


Cloning into '/kaggle/working/roco-spring-devkit'...
Note: switching to '90ae81a9324c6806dc3c2482aab84a2744215bd9'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 90ae81a Merge pull request #3 from hmorimitsu/fixes2


Obtaining file:///kaggle/working/roco-spring-devkit
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 

2026-09-11 13:47:42.431 | WARNING  | roco_spring_devkit.scene_flow.models.raft_3d.raft_3d:<module>:12 - lietorch not found. Please install lietorch from https://github.com/princeton-vl/lietorch if you want to use RAFT3D
2026-09-11 13:47:42.454 | WARNING  | roco_spring_devkit.scene_flow.models.raft_3d.blocks.grid:<module>:13 - scikit-sparse not found. Please install scikit-sparse if you want to use RAFT3D.


Devkit ready; full RAFT has 5.26M parameters


## 3. Fine-tune from the supplied Sintel checkpoint

The official `train.py` and `FlowDataModule` are used. `spring-train-left` consumes only the available left forward-flow labels, while `spring-val-left` selects sequence `0022`. The checkpoint connector restores the supplied Sintel model weights before optimization. Training uses all-pairs correlation, the official training mode; Triton correlation is selected later for memory-efficient native inference.


In [4]:
training_config_path = WORK_ROOT / "raft_spring_left_finetune.yaml"
training_config = f'''# Generated by raft-spring-finetune-inference.ipynb
seed_everything: {SEED}
ckpt_path: {PRETRAINED_CHECKPOINT}
lr: {TRAIN_LR}
wdecay: {TRAIN_WEIGHT_DECAY}
logger: tensorboard
log_dir: {TRAIN_LOG_DIR}
train_ckpt_topk: 0
infer_ckpt_topk: 1
infer_ckpt_metric: main_val_metric
trainer:
  accelerator: gpu
  devices: 1
  precision: {TRAIN_PRECISION}
  max_epochs: {TRAIN_EPOCHS}
  accumulate_grad_batches: {ACCUMULATE_GRAD_BATCHES}
  gradient_clip_val: 1.0
  log_every_n_steps: 10
  num_sanity_val_steps: 2
model:
  class_path: raft
  init_args:
    iters: {TRAIN_ITERS}
    corr_mode: allpairs
    predict_all_directions: false
data:
  train_dataset: spring-train-left
  val_dataset: spring-val-left
  spring_root_dir: {TRAIN_SPRING_ROOT}
  train_batch_size: {TRAIN_BATCH_SIZE}
  train_num_workers: 2
  train_crop_size: [{TRAIN_CROP_SIZE[0]}, {TRAIN_CROP_SIZE[1]}]
  train_transform_cuda: false
  train_transform_fp16: false
'''
training_config_path.write_text(training_config)
print(training_config)

train_env = os.environ.copy()
train_env["CUDA_VISIBLE_DEVICES"] = "0"
train_env["PYTHONUNBUFFERED"] = "1"
train_env["PYTHONPATH"] = devkit_path + (os.pathsep + train_env["PYTHONPATH"] if train_env.get("PYTHONPATH") else "")

if RUN_TRAINING:
    subprocess.run(
        [sys.executable, "train.py", "--config", str(training_config_path)],
        cwd=DEVKIT_DIR / "roco_spring_devkit/optical_flow",
        env=train_env,
        check=True,
    )
    best = sorted(TRAIN_LOG_DIR.rglob("raft_best_*.ckpt"), key=lambda p: p.stat().st_mtime)
    fallback = sorted(TRAIN_LOG_DIR.rglob("raft_last_*.ckpt"), key=lambda p: p.stat().st_mtime)
    candidates = best or fallback
    if not candidates:
        raise RuntimeError(f"Training finished but no checkpoint was found under {TRAIN_LOG_DIR}")
    shutil.copy2(candidates[-1], FINETUNED_CHECKPOINT)
    print("Selected checkpoint:", candidates[-1])
    print("Portable checkpoint:", FINETUNED_CHECKPOINT)

print("Training output checkpoint:", FINETUNED_CHECKPOINT)


# Generated by raft-spring-finetune-inference.ipynb
seed_everything: 3407
ckpt_path: /kaggle/input/datasets/strikingratio/ckpoint/raft-sintel-fb44381e.ckpt
lr: 1e-05
wdecay: 1e-05
logger: tensorboard
log_dir: /kaggle/working/raft_spring_train_logs
train_ckpt_topk: 0
infer_ckpt_topk: 1
infer_ckpt_metric: main_val_metric
trainer:
  accelerator: gpu
  devices: 1
  precision: 16-mixed
  max_epochs: 1
  accumulate_grad_batches: 4
  gradient_clip_val: 1.0
  log_every_n_steps: 10
  num_sanity_val_steps: 2
model:
  class_path: raft
  init_args:
    iters: 12
    corr_mode: allpairs
    predict_all_directions: false
data:
  train_dataset: spring-train-left
  val_dataset: spring-val-left
  spring_root_dir: /kaggle/input/datasets/sakhawatdhrubo2/spring-train-left-10seq/spring_subset/spring
  train_batch_size: 1
  train_num_workers: 2
  train_crop_size: [540, 960]
  train_transform_cuda: false
  train_transform_fp16: false



2026-09-11 13:47:51.347 | WARNING  | roco_spring_devkit.scene_flow.models.raft_3d.raft_3d:<module>:12 - lietorch not found. Please install lietorch from https://github.com/princeton-vl/lietorch if you want to use RAFT3D
2026-09-11 13:47:51.351 | WARNING  | roco_spring_devkit.scene_flow.models.raft_3d.blocks.grid:<module>:13 - scikit-sparse not found. Please install scikit-sparse if you want to use RAFT3D.
/kaggle/working/roco-spring-devkit/roco_spring_devkit/common/data/optical_flow_transforms.py:142: SyntaxWarning: invalid escape sequence '\|'
  In other words, a pixel p is considered occluded when \|Ff(p) + Fb(p + F(f))\|_2 > threshold,
/kaggle/working/roco-spring-devkit/roco_spring_devkit/common/data/optical_flow_transforms.py:160: SyntaxWarning: invalid escape sequence '\|'
  A pixel is considered occluded if \|Ff(p) + Fb(p + F(f))\|_2 > threshold.
Seed set to 3407
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU 

┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss_fn       │ SequenceLoss     │      0 │ train │     0 │
│ 1 │ train_metrics │ FlowMetrics      │      0 │ train │     0 │
│ 2 │ val_metrics   │ ModuleList       │      0 │ train │     0 │
│ 3 │ fnet          │ BasicEncoder     │  1.1 M │ train │     0 │
│ 4 │ cnet          │ BasicEncoder     │  1.1 M │ train │     0 │
│ 5 │ update_block  │ BasicUpdateBlock │  3.1 M │ train │     0 │
└───┴───────────────┴──────────────────┴────────┴───────┴───────┘
Trainable params: 5.3 M                                                         
Non-trainable params: 0                                                         
Total params: 5.3 M                                                             
Total estimated model params size (MB): 21.030                                  
Modules in train

09/11/2026 13:48:10 - INFO: The provided checkpoint does not contain the training state. Only the model weights will be loaded.
Restored all states from the checkpoint at /kaggle/input/datasets/strikingratio/ckpoint/raft-sintel-fb44381e.ckpt
2026-09-11 13:48:10.341 | INFO     | roco_spring_devkit.common.data.optical_flow_datasets:_log_status:257 - Loading 18 samples from Spring dataset.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size

Epoch 0/0  ━━━━━━━━━━━━━━━━ 856/856 0:10:21 • 0:00:00 1.45it/s v_num: 0.000     
                                                               epe_step: 0.124  
                                                               spring-val: 0.816
                                                               epe_epoch: 1.469 
Selected checkpoint: /kaggle/working/raft_spring_train_logs/raft-spring-11092026_134752/raft-spring-train-left/version_0/checkpoints/raft_best_main_val_metric=0.82_epoch=0_step=214.ckpt
Portable checkpoint: /kaggle/working/raft-spring-left-finetuned.ckpt
Training output checkpoint: /kaggle/working/raft-spring-left-finetuned.ckpt


## Training handoff

Download or publish `/kaggle/working/raft-spring-left-finetuned.ckpt` as a private Kaggle Dataset. The inference notebook searches attached inputs for this exact filename. Keep the training logs if you want to compare validation metrics across longer runs.

Start with one epoch. Increase `TRAIN_EPOCHS` only after this run succeeds and only while the held-out `0022` metric improves.
